# Farm Credit Risk Screening — Educational Classification and Explainable AI

**Independent educational portfolio project; not affiliated with or endorsed by the Farm Credit Administration.**

This notebook demonstrates Logistic Regression, Random Forest, MLflow tracking, and SHAP attribution using public FCA quarterly financial indicators. **The target labels are constructed by an illustrative rule, NOT observed distress, default, or FCA supervisory outcomes.** Evaluation measures fidelity to that rule only.

## Input and reproducibility

- Source: corrected `model_comparison.csv` generated by `real_fca_analysis.py` followed by `validate_models.py` in the companion [FCA Financial Risk Analytics repository](https://github.com/AJ277/farm-credit-financial-risk-analytics).
- Input location in this workspace: `/Volumes/workspace/usda_training/raw_files/model_comparison.csv`. Upload the regenerated CSV there before running the notebook; the file is not bundled with this notebook.
- Output Delta table: `workspace.usda_training.fca_financial_metrics`. If it already exists, the notebook reuses it without replacing it. Verify the table corresponds to the intended input before running.
- Uses 177 observations from 59 institutions over 2026 Q1–Q3. Four financial indicators are used for classification.
- Q1 thresholds are calculated from the **full Q1 sample, including institutions that later appear in the holdout**. This is a limitation in independence of the label-construction procedure. The test metrics should not be interpreted as an independent generalization test of risk outcomes.
- The label is 1 when Q1-calibrated nonaccrual ratio is at least the Q1 75th percentile **or** equity/assets is at most the Q1 25th percentile; otherwise 0. These are illustrative thresholds, not FCA standards.
- Institution-grouped 80/20 train/test split avoids sharing the same institution across model training and test observations. The scaler fits only training observations.
- SHAP describes the fitted model's contributions and does not establish causation.

**Environment:** Databricks with PySpark, pandas, scikit-learn, MLflow, and SHAP. Run installation cell before other Python cells when needed; after a Python restart, run the notebook from the beginning.


In [ ]:
%pip install shap


In [ ]:

import sys
import pandas as pd
import sklearn
import mlflow

from pyspark.sql import functions as F
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print("FCA CREDIT RISK MACHINE LEARNING PROJECT")
print("----------------------------------------")

print("Python:", sys.version.split()[0])
print("Spark:", spark.version)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("MLflow:", mlflow.__version__)

print("----------------------------------------")
print("FCA ML ENVIRONMENT READY")


In [ ]:

# Load the corrected FCA financial dataset

csv_path = (
    "/Volumes/workspace/usda_training/raw_files/"
    "model_comparison.csv"
)

fca_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(csv_path)
)

print("FCA FINANCIAL DATA VALIDATION")
print("-----------------------------")

print("Total rows:", fca_df.count())
print("Total columns:", len(fca_df.columns))

print("\nColumn names:")
for column in fca_df.columns:
    print(column)

print("Input ready for validation")


In [ ]:

# Create the FCA Delta table without overwriting existing data

table_name = "workspace.usda_training.fca_financial_metrics"

if spark.catalog.tableExists(table_name):
    print("Table already exists. No changes made.")
else:
    (
        fca_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
    print("Delta table created successfully.")

verified_df = spark.table(table_name)

print("Verified rows:", verified_df.count())
print("Verified columns:", len(verified_df.columns))


In [ ]:

from pyspark.sql import functions as F

# Load the saved Delta table
verified_df = spark.table(
    "workspace.usda_training.fca_financial_metrics"
)

features = [
    "loan_growth_qoq_pct",
    "equity_assets_pct",
    "nonaccrual_netloans_pct",
    "annualized_quarterly_income_avg_assets_pct"
]

print("FCA FINANCIAL FEATURE VALIDATION")
print("--------------------------------")

print("Total observations:", verified_df.count())

# Check for duplicate institution-quarter records
duplicates = (
    verified_df
    .groupBy("institution_id", "quarter")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate institution-quarter keys:", duplicates)

# Check each feature for missing or invalid values
for feature in features:
    missing = verified_df.filter(
        F.col(feature).isNull() |
        F.isnan(F.col(feature))
    ).count()

    print(f"{feature} missing values: {missing}")

# Show descriptive statistics
display(verified_df.select(*features).summary())

print("FEATURE VALIDATION COMPLETED")


In [ ]:

from pyspark.sql import functions as F

df = spark.table(
    "workspace.usda_training.fca_financial_metrics"
)

features = [
    "loan_growth_qoq_pct",
    "equity_assets_pct",
    "nonaccrual_netloans_pct",
    "annualized_quarterly_income_avg_assets_pct"
]

print("FCA FINANCIAL FEATURE DISTRIBUTIONS")
print("-----------------------------------")

# Display distribution statistics for each indicator
display(
    df.select(*features).summary(
        "count",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    )
)

# Review the number of institutions and quarters
print(
    "Unique institutions:",
    df.select("institution_id").distinct().count()
)

print("Observations by quarter:")
display(
    df.groupBy("quarter")
      .count()
      .orderBy("quarter")
)

print("FEATURE DISTRIBUTION ANALYSIS COMPLETED")


In [ ]:

from pyspark.sql import functions as F

df = spark.table(
    "workspace.usda_training.fca_financial_metrics"
)

# Confirm the number of distinct institutions
print("Unique institutions:",
      df.select("institution_id").distinct().count())

# Calculate illustrative thresholds from full Q1 (including held-out institutions).
# These labels are rule-derived, not independent risk outcomes.
baseline = df.filter(F.col("quarter") == "2026Q1")

nonaccrual_threshold = baseline.approxQuantile(
    "nonaccrual_netloans_pct", [0.75], 0.0
)[0]

equity_threshold = baseline.approxQuantile(
    "equity_assets_pct", [0.25], 0.0
)[0]

print("Nonaccrual threshold:", nonaccrual_threshold)
print("Equity threshold:", equity_threshold)

# 1 means meeting at least one illustrative
# financial-screening condition.
df_labeled = df.withColumn(
    "educational_screening_label",
    (
        (F.col("nonaccrual_netloans_pct") >=
         F.lit(nonaccrual_threshold))
        |
        (F.col("equity_assets_pct") <=
         F.lit(equity_threshold))
    ).cast("integer")
)

print("\nEducational label distribution:")
display(
    df_labeled.groupBy(
        "quarter", "educational_screening_label"
    ).count().orderBy(
        "quarter", "educational_screening_label"
    )
)

print("EDUCATIONAL CLASSIFICATION LABELS CREATED")


In [ ]:

import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# Use only the four original financial indicators.
features = [
    "loan_growth_qoq_pct",
    "equity_assets_pct",
    "nonaccrual_netloans_pct",
    "annualized_quarterly_income_avg_assets_pct"
]

ml_df = df_labeled.select(
    "institution_id",
    "quarter",
    *features,
    "educational_screening_label"
).toPandas()

# Keep institutions together during splitting.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        ml_df[features],
        ml_df["educational_screening_label"],
        groups=ml_df["institution_id"]
    )
)

X_train = ml_df.iloc[train_idx][features].copy()
X_test = ml_df.iloc[test_idx][features].copy()

y_train = ml_df.iloc[train_idx][
    "educational_screening_label"
].astype(int)

y_test = ml_df.iloc[test_idx][
    "educational_screening_label"
].astype(int)

# Standardize using training data only.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

train_institutions = set(
    ml_df.iloc[train_idx]["institution_id"]
)

test_institutions = set(
    ml_df.iloc[test_idx]["institution_id"]
)

overlap = train_institutions & test_institutions

print("FCA MACHINE LEARNING DATA SPLIT")
print("--------------------------------")
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Training institutions:", len(train_institutions))
print("Testing institutions:", len(test_institutions))
print("Institution overlap:", len(overlap))

print("\nTraining labels:")
print(y_train.value_counts().sort_index())

print("\nTesting labels:")
print(y_test.value_counts().sort_index())

assert len(overlap) == 0, "Institution leakage detected"
assert len(set(y_train)) == 2, "Training split lacks a class"
assert len(set(y_test)) == 2, "Test split lacks a class"

print("\nDATA SPLIT VALIDATION PASSED")


In [ ]:

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Initialize classification models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():

    # Random Forest does not require scaling
    if name == "Logistic Regression":
        train_data = X_train_scaled
        test_data = X_test_scaled
    else:
        train_data = X_train
        test_data = X_test

    model.fit(train_data, y_train)
    trained_models[name] = model

    predictions = model.predict(test_data)
    probabilities = model.predict_proba(test_data)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test, probabilities
        )
    })

    print(f"\n{name}")
    print("---------------------------")
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))

results_df = pd.DataFrame(results)

print("\nFCA CLASSIFICATION MODEL COMPARISON")
print("-----------------------------------")
display(results_df.round(4))

print("MODEL TRAINING COMPLETED")


In [ ]:

import mlflow

# Build the required absolute Databricks workspace path
username = spark.sql(
    "SELECT current_user()"
).first()[0]

experiment_path = (
    f"/Users/{username}/FCA_Credit_Risk_Educational"
)

print("MLflow experiment path configured for current Databricks user")

# Set the correct experiment
experiment = mlflow.set_experiment(experiment_path)

print("Experiment ID:", experiment.experiment_id)

# Log both trained models' results
for name, model in trained_models.items():

    row = results_df.loc[
        results_df["Model"] == name
    ].iloc[0]

    with mlflow.start_run(
        run_name=name.replace(" ", "_")
    ):
        mlflow.log_params(model.get_params())

        for metric in [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC_AUC"
        ]:
            mlflow.log_metric(
                metric.lower(),
                float(row[metric])
            )

        mlflow.set_tags({
            "project": "FCA Educational Risk Screening",
            "label_type": "Rule-derived educational",
            "validation": "Institution-grouped holdout",
            "limitation": (
                "Not validated for actual "
                "institutional distress or default"
            )
        })

        print("Logged:", name)

print("\nMLFLOW EXPERIMENT TRACKING COMPLETED")


In [ ]:

import shap

print("SHAP version:", shap.__version__)
print("SHAP ENVIRONMENT READY")


In [ ]:

import numpy as np
import pandas as pd
import shap

# Retrieve the trained Random Forest
rf_model = trained_models["Random Forest"]

# Calculate SHAP values for the test observations
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# Extract contributions for educational Label 1
if isinstance(shap_values, list):
    label1_values = shap_values[1]
elif np.asarray(shap_values).ndim == 3:
    label1_values = shap_values[:, :, 1]
else:
    label1_values = shap_values

# Rank financial indicators by average contribution
importance = pd.DataFrame({
    "Financial Indicator": X_test.columns,
    "Mean Absolute SHAP": np.abs(label1_values).mean(axis=0)
}).sort_values(
    "Mean Absolute SHAP",
    ascending=False
)

print("FCA RANDOM FOREST SHAP FEATURE IMPORTANCE")
print("-----------------------------------------")
display(importance)

print("SHAP ANALYSIS COMPLETED")


In [ ]:

import matplotlib.pyplot as plt

# Use the importance DataFrame from Lesson 10
plot_data = importance.sort_values(
    "Mean Absolute SHAP",
    ascending=True
)

fig, ax = plt.subplots(figsize=(11, 5.5))

ax.barh(
    plot_data["Financial Indicator"],
    plot_data["Mean Absolute SHAP"],
    color="steelblue"
)

ax.set_title(
    "FCA Educational Risk Screening\n"
    "Random Forest SHAP Feature Importance",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Mean Absolute SHAP Value")
ax.set_ylabel("Financial Indicator")

ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()

# Save the chart for later GitHub documentation
chart_path = "/tmp/fca_shap_feature_importance.png"

plt.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

print("Chart saved:", chart_path)
print("SHAP VISUALIZATION COMPLETED")


In [ ]:

from pathlib import Path
import shutil

# Persistent destination in Unity Catalog
output_dir = Path(
    "/Volumes/workspace/usda_training/"
    "raw_files/fca_ml_outputs"
)

output_dir.mkdir(parents=True, exist_ok=True)

source_chart = Path(
    "/tmp/fca_shap_feature_importance.png"
)

destination_chart = (
    output_dir / "fca_shap_feature_importance.png"
)

shutil.copy2(
    source_chart,
    destination_chart
)

print("Chart copied to:", destination_chart)
print("File exists:", destination_chart.exists())
print("File size:", destination_chart.stat().st_size)
print("SHAP CHART BACKUP COMPLETED")


In [ ]:

from pathlib import Path

output_dir = Path(
    "/Volumes/workspace/usda_training/"
    "raw_files/fca_ml_outputs"
)

output_dir.mkdir(parents=True, exist_ok=True)

# Export the model evaluation results
report_path = output_dir / "fca_model_comparison.csv"

results_df.to_csv(report_path, index=False)

print("Model report:", report_path)
print("File exists:", report_path.exists())

print("\nMODEL EVALUATION RESULTS")
print(results_df.to_string(index=False))

print("\nMODEL COMPARISON EXPORT COMPLETED")


In [ ]:

limitations = """FCA EDUCATIONAL RISK CLASSIFICATION
===================================

PROJECT PURPOSE
Demonstrate classification, model evaluation,
MLflow experiment tracking, and SHAP explanations.

DATA
177 observations representing 59 institutions
across 2026 Q1-Q3.

EDUCATIONAL LABELS
Labels are derived from illustrative financial
ratio screening thresholds calculated from
2026 Q1 observations.

The classification target is not a verified
default, distress, or supervisory outcome.

MODEL VALIDATION
141 training observations from 47 institutions.
36 testing observations from 12 institutions.
No institution overlap between sets.

IMPORTANT LIMITATIONS
- Model evaluation measures reproduction of a
  rule-derived educational classification.
- High accuracy does not establish the ability
  to predict real institutional financial distress.
- SHAP explains model behavior, not causation.
- The evaluation uses one grouped holdout split.
- Results are not FCA examination findings.
- This is an independent educational project,
  not affiliated with or endorsed by the FCA.
"""

limitations_path = (
    output_dir / "PROJECT_LIMITATIONS.txt"
)

limitations_path.write_text(
    limitations,
    encoding="utf-8"
)

print("Limitations file:", limitations_path)
print("File exists:", limitations_path.exists())
print("PROJECT DOCUMENTATION COMPLETED")


In [ ]:

print("FCA MACHINE LEARNING PROJECT FILES")
print("----------------------------------")

for file in sorted(output_dir.iterdir()):
    print(file.name, file.stat().st_size, "bytes")

print("PROJECT EXPORT VERIFICATION COMPLETED")
